# Exp 17: Background paste

Card: [exp17_background_paste.md](../../experiments/exp17_background_paste.md)


In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.augment import RECIPES, Validation
from core.config import CLASSES, MEAN, ROOT, SEED, STD, TRAIN_DIR
from core.cv import VIP, CVResult, Config, CrossValidator
from core.data import KaggleDataModule
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-background-paste-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_IMAGES = 1  # rows per exp13 figure
    MAX_DRAWS = 2  # columns of draws
    N_POSITIVES = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_IMAGES = 8
    MAX_DRAWS = None  # all draws
    N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp17_background_paste" / "resnet18.toml", **OVERRIDES
)
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")
FOLD = 0
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp17_background_paste"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "background paste": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Oversampled views (exp13)


In [ ]:
torch.manual_seed(SEED)
dm = KaggleDataModule(
    CONFIG.read_data(),
    fold=FOLD,
    batch_size=CONFIG.batch_size,
    num_workers=0,
    train_tfms=RECIPES[CONFIG.augment](CONFIG.img_size).compose(),
    oversample=CONFIG.oversample,
    downsample=CONFIG.downsample,
)
dm.setup()
train = dm.train_ds.df
draws = Counter(dm.train_dataloader().sampler)  # dataset row: times drawn in epoch


def show(x):
    """Normalized CHW tensor back to HWC image in [0, 1]."""
    return (x.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)


resize = Validation(CONFIG.img_size).compose()
rng = np.random.default_rng(SEED)
figures = {}
for cls in VIP:
    rows = rng.choice(train.index[train[cls] == 1], N_IMAGES, replace=False)
    n_cols = 1 + (MAX_DRAWS or max(draws[i] for i in rows))
    fig, axes = plt.subplots(
        N_IMAGES, n_cols, figsize=(1.8 * n_cols, 2.0 * N_IMAGES), squeeze=False
    )
    for ax_row, i in zip(axes, rows):
        image = np.array(
            Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg").convert("RGB")
        )
        ax_row[0].imshow(show(resize(image=image)["image"]))
        ax_row[0].set_title(f"original · {draws[i]} draws", fontsize=8)
        for ax in ax_row[1 : 1 + min(draws[i], n_cols - 1)]:
            ax.imshow(
                show(dm.train_ds[i][0])
            )  # fresh augmentation per draw, as in training
        for ax in ax_row:
            ax.set_axis_off()
    fig.suptitle(
        f"{cls} · fold {FOLD} train · one epoch, oversample {CONFIG.oversample}, "
        f"downsample {CONFIG.downsample}, augment {CONFIG.augment!r}"
    )
    fig.tight_layout()
    figures[f"{PREFIX}_views_{cls.lower()}.png"] = fig
plt.show()

## Training


In [ ]:
results = {"exp16": CVResult.load(EXP16)}
results["background paste"] = CrossValidator(CONFIG).run()

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
cams = GradCamFigures(CONFIG)
for cls in VIP:
    for train in (False, True):
        fig = cams.positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)